# Low-Data Current-Local Confirmation

Use this notebook in Google Colab for the May 2026 low-data confirmation run.

This is not a V7 run. It trains only:

- `classical_conv`
- `non_trainable_quantum`

It confirms the seed-42 low-data pilot with seeds `43` and `44` across train fractions `0.10`, `0.25`, `0.50`, and `1.00`.

Before running this notebook, commit and push the local low-data implementation changes. Colab clones from GitHub, so it cannot see uncommitted local files.

## 1. Runtime Check

Use a GPU runtime. L4 is enough. A100 is acceptable but not required.

In [ ]:
!nvidia-smi

import torch
print('torch:', torch.__version__)
print('cuda:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))


## 2. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 3. Clone Or Refresh Repo

If this cell says `scripts/run_low_data_grid.py` is missing later, stop and push the local commit first.

In [ ]:
import os

REPO_URL = 'https://github.com/necatiincekara/Quanvolutional-Neural-Network.git'
BRANCH = 'master'
REPO_PATH = '/content/Quanvolutional-Neural-Network'

if not os.path.exists(REPO_PATH):
    !git clone {REPO_URL} {REPO_PATH}

%cd {REPO_PATH}
!git fetch origin
!git checkout {BRANCH}
!git pull --ff-only origin {BRANCH}
!git log --oneline -n 5


## 4. Install Dependencies

In [ ]:
%cd /content/Quanvolutional-Neural-Network
!pip install -r requirements.txt


## 5. Verify Code And Dataset

Dataset is expected at:

- `/content/drive/MyDrive/set/train`
- `/content/drive/MyDrive/set/test`

In [ ]:
%cd /content/Quanvolutional-Neural-Network
!test -f scripts/run_low_data_grid.py || (echo 'Missing scripts/run_low_data_grid.py. Commit and push local changes, then rerun clone/pull cell.' && false)
!test -f scripts/aggregate_low_data.py || (echo 'Missing scripts/aggregate_low_data.py. Commit and push local changes, then rerun clone/pull cell.' && false)
!test -d /content/drive/MyDrive/set/train || (echo 'Missing Drive train set: /content/drive/MyDrive/set/train' && false)
!test -d /content/drive/MyDrive/set/test || (echo 'Missing Drive test set: /content/drive/MyDrive/set/test' && false)
!find /content/drive/MyDrive/set/train -type f | wc -l
!find /content/drive/MyDrive/set/test -type f | wc -l


## 6. Smoke Checks

This should be quick. If anything fails here, do not start the confirmation run.

In [ ]:
%cd /content/Quanvolutional-Neural-Network
!python -m py_compile src/benchmark_protocol.py train_ablation_local.py train_thesis_models.py scripts/run_low_data_grid.py scripts/aggregate_low_data.py
!python train_ablation_local.py --model classical_conv --test
!python train_thesis_models.py --model thesis_hqnn2 --test
!python scripts/run_low_data_grid.py --protocol-version low_data_confirm_v1 --models classical_conv non_trainable_quantum --fractions 0.10 --seeds 43 --split-seed 42 --device auto


## 7. Run Confirmation Training

This is the main training cell. Run this cell once. Result JSONs and checkpoints are written directly into Drive through symlinked artifact directories.

In [ ]:
%cd /content/Quanvolutional-Neural-Network
!RUN_DIR=/content/drive/MyDrive/quanv_results/low_data_confirm_20260502; \
  mkdir -p "$RUN_DIR/experiments_low_data" "$RUN_DIR/models_low_data" experiments models; \
  if [ -d experiments/low_data ] && [ ! -L experiments/low_data ]; then rsync -a experiments/low_data/ "$RUN_DIR/experiments_low_data/"; fi; \
  rm -rf experiments/low_data models/low_data; \
  ln -s "$RUN_DIR/experiments_low_data" experiments/low_data; \
  ln -s "$RUN_DIR/models_low_data" models/low_data; \
  ls -ld experiments/low_data models/low_data
!python scripts/run_low_data_grid.py \
  --execute \
  --protocol-version low_data_confirm_v1 \
  --models classical_conv non_trainable_quantum \
  --fractions 0.10 0.25 0.50 1.00 \
  --seeds 43 44 \
  --split-seed 42 \
  --device auto


## 8. Aggregate And Copy Artifacts To Drive

Run this after training completes. If the browser disconnected but the runtime is still alive, reconnect and run this cell. If the runtime was fully reset, first rerun sections 2 through 4, then run this cell.

In [ ]:
%cd /content/Quanvolutional-Neural-Network
!RUN_DIR=/content/drive/MyDrive/quanv_results/low_data_confirm_20260502; \
  mkdir -p "$RUN_DIR/experiments_low_data" "$RUN_DIR/models_low_data" experiments models; \
  rm -rf experiments/low_data models/low_data; \
  ln -s "$RUN_DIR/experiments_low_data" experiments/low_data; \
  ln -s "$RUN_DIR/models_low_data" models/low_data
!python scripts/aggregate_low_data.py
!RUN_DIR=/content/drive/MyDrive/quanv_results/low_data_confirm_20260502; \
  mkdir -p "$RUN_DIR/experiments_low_data" "$RUN_DIR/docs"; \
  rsync -a experiments/low_data/ "$RUN_DIR/experiments_low_data/"; \
  cp experiments/low_data_summary.json "$RUN_DIR/"; \
  cp docs/LOW_DATA_SUMMARY.md "$RUN_DIR/docs/"; \
  ls -lah "$RUN_DIR"; \
  ls -lah "$RUN_DIR/experiments_low_data" | tail -40


## 9. Inspect Decision Rows

In [ ]:
import json
from pathlib import Path

data = json.loads(Path('experiments/low_data_summary.json').read_text())
for row in data['comparisons']:
    print(
        row['family'],
        'frac=', row['train_fraction'],
        row['quantum_model'], row['quantum_test_acc'],
        'vs', row['classical_model'], row['classical_test_acc'],
        'gap C-Q=', row['gap_classical_minus_quantum'],
        'signal=', row['colab_signal'],
        row['signal_reason'],
    )
